# Weighted Lebesgue Spaces and Bessel-Sobolev Priors

This notebook demonstrates the new `WeightedLebesgue` space and how inner-product weighting affects
Laplacian-based priors via `BesselSobolevInverse` covariances.

In [9]:
import numpy as np
import matplotlib.pyplot as plt
from intervalinf import IntervalDomain, Function
from intervalinf.spaces import Lebesgue, WeightedLebesgue
from intervalinf.core.boundary import BoundaryConditions
from intervalinf.core.config import IntegrationConfig
from intervalinf.operators import Laplacian, BesselSobolevInverse

print("Imports successful!")

Imports successful!


## 1. Create Plain and Weighted Lebesgue Spaces

In [10]:
domain = IntervalDomain(0.1, 1.0)
bc = BoundaryConditions.dirichlet()
integration_config = IntegrationConfig(method="simpson", n_points=2000)

# Plain L² space
space_plain = Lebesgue(0, domain, basis=None, integration_config=integration_config)

# Weighted L²(r²) space
def weight_r2(r):
    return np.asarray(r, dtype=float) ** 2

space_weighted = WeightedLebesgue(0, domain, weight_r2, integration_config=integration_config)

print(f"Plain L² space created")
print(f"Weighted L²(r²) space created")

Plain L² space created
Weighted L²(r²) space created


## 2. Compare Inner Products

In [11]:
f = Function(domain, evaluate_callable=lambda r: np.sin(np.pi * np.asarray(r)))
g = Function(domain, evaluate_callable=lambda r: np.cos(2 * np.pi * np.asarray(r)))

inner_plain = space_plain.inner_product(f, g)
inner_weighted = space_weighted.inner_product(f, g)

print(f"⟨f, g⟩ (plain L²): {inner_plain:.6f}")
print(f"⟨f, g⟩_w (weighted): {inner_weighted:.6f}")
print(f"Difference: {abs(inner_plain - inner_weighted):.6f}")

⟨f, g⟩ (plain L²): -0.226286
⟨f, g⟩_w (weighted): -0.044057
Difference: 0.182229


## 3. Create Laplacians

In [12]:
# Laplacian on plain L²
L_plain = Laplacian(space_plain, bc, 1.0, method="spectral", dofs=20, integration_config=integration_config)

# Laplacian on weighted L²(r²)
L_weighted = Laplacian(space_weighted, bc, 1.0, method="spectral", dofs=20, integration_config=integration_config)

print("Laplacians created.")
print(f"\nFirst 3 eigenvalues:")
for i in range(3):
    print(f"  λ_{i}: {L_plain.get_eigenvalue(i):.4f}")

Laplacians created.

First 3 eigenvalues:
  λ_0: 12.1847
  λ_1: 48.7388
  λ_2: 109.6623


## 4. Create Bessel-Sobolev Covariances

In [13]:
k, s = 1.5, 1.0

C_plain = BesselSobolevInverse(domain=space_plain, codomain=space_plain, k=k, s=s, 
    L=L_plain, dofs=20, integration_config=integration_config)

C_weighted = BesselSobolevInverse(domain=space_weighted, codomain=space_weighted, k=k, s=s, 
    L=L_weighted, dofs=20, integration_config=integration_config)

print(f"Bessel-Sobolev C = (k² I - Δ)^{{-s}} created")
print(f"  k={k}, s={s}")
print(f"  Plain: uses fast transforms = {C_plain._can_use_fast_transforms}")
print(f"  Weighted: uses radial fast path = {C_weighted._radial_dirichlet_fast}")

Bessel-Sobolev C = (k² I - Δ)^{-s} created
  k=1.5, s=1.0
  Plain: uses fast transforms = True
  Weighted: uses radial fast path = False


## 5. Compare Prior Standard Deviations

In [ ]:
r_eval = np.linspace(0.15, 0.95, 50)

# Compare how the covariance operators scale different functions
f_test = Function(domain, evaluate_callable=lambda r: np.sin(np.pi * (np.asarray(r) - 0.1) / 0.9))

Cf_plain = C_plain(f_test)
Cf_weighted = C_weighted(f_test)

# Evaluate both at the test points
Cf_plain_vals = np.array([Cf_plain(r) for r in r_eval])
Cf_weighted_vals = np.array([Cf_weighted(r) for r in r_eval])

# The prior variance is related to C applied to a delta-like function
# For visualization, we'll show how the covariance extends the effect of a localized function

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Original test function
f_vals = np.array([f_test(r) for r in r_eval])
axes[0, 0].plot(r_eval, f_vals, 'k-', linewidth=2)
axes[0, 0].set_title('Test Function: sin(π(r - 0.1) / 0.9)')
axes[0, 0].set_ylabel('f(r)')
axes[0, 0].grid(True, alpha=0.3)

# Covariance applied to test function
axes[0, 1].plot(r_eval, Cf_plain_vals, 'b-', linewidth=2, label='Plain L²')
axes[0, 1].plot(r_eval, Cf_weighted_vals, 'r--', linewidth=2, label='Weighted L²(r²)')
axes[0, 1].set_title('Covariance Applied: C f(r)')
axes[0, 1].set_ylabel('(C f)(r)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Weighted inner product of test function with itself
norm_plain = space_plain.inner_product(f_test, f_test)
norm_weighted = space_weighted.inner_product(f_test, f_test)
axes[1, 0].bar(['Plain L²', 'Weighted L²(r²)'], [norm_plain, norm_weighted], color=['b', 'r'])
axes[1, 0].set_title(f'‖f‖² (Inner Product of Test Function)')
axes[1, 0].set_ylabel('‖f‖²')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Prior variance (diagonal of covariance): ⟨C δ_r, δ_r⟩ ≈ C applied to narrow bump
prior_var_plain = np.array([space_plain.inner_product(C_plain(Function(domain, lambda x: 1.0 if abs(x-r) < 0.02 else 0)), 
                                                        Function(domain, lambda x: 1.0 if abs(x-r) < 0.02 else 0)) for r in r_eval[:20]])
prior_var_weighted = np.array([space_weighted.inner_product(C_weighted(Function(domain, lambda x: 1.0 if abs(x-r) < 0.02 else 0)),
                                                            Function(domain, lambda x: 1.0 if abs(x-r) < 0.02 else 0)) for r in r_eval[:20]])

axes[1, 1].plot(r_eval[:20], np.sqrt(np.abs(prior_var_plain)), 'b-', linewidth=2, marker='o', label='Plain L²')
axes[1, 1].plot(r_eval[:20], np.sqrt(np.abs(prior_var_weighted)), 'r--', linewidth=2, marker='s', label='Weighted L²(r²)')
axes[1, 1].set_xlabel('r')
axes[1, 1].set_ylabel('Prior Std Dev')
axes[1, 1].set_title('Approximate Prior Standard Deviation')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Inner products of test function:")
print(f"  ⟨f, f⟩_plain = {norm_plain:.6f}")
print(f"  ⟨f, f⟩_weighted = {norm_weighted:.6f}")
print(f"  Ratio (weighted/plain) = {norm_weighted/norm_plain:.4f}")
print(f"\nWeighting effect: The weighted inner product is smaller due to w(r) = r²")
print(f"being small near the lower boundary (r ≈ 0.1).")

## Summary

✅ **WeightedLebesgue** implements $L^2(w)$ as a unified mechanism via `MassWeightedHilbertSpace`

✅ **Weighted priors** automatically encode geometric structure (e.g., spherical shells with $w(r)=r^2$)

✅ **Transparent to operators** — the same `BesselSobolevInverse` code handles both plain and weighted domains

✅ **Self-adjoint** on the space's inner product, whether plain or weighted